In [1]:
# %%
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

PATH_INDICES = 'csv_dashboard/BaseINDICES-2020-2025.csv'
PATH_ING_CHILE = 'csv_dashboard/clean_kpis.csv'

COL_INST_INDICES = 'Nombre Institución'
COL_REG_INDICES = 'Nombre Region'
COL_CARRERA_INDICES = 'Carrera Genérica'

# 1. Verificación de existencia de archivos en el directorio local
for ruta_archivo in [PATH_INDICES, PATH_ING_CHILE]:
    if not os.path.exists(ruta_archivo):
        raise FileNotFoundError(f"No se encontro el archivo: '{ruta_archivo}'.")

print("Cargando matrices de datos en memoria...")

# 2. Lectura con fallback de codificación para evitar errores de parseo
try:
    df_indices = pd.read_csv(PATH_INDICES, sep=';', encoding='utf-8')
except Exception:
    df_indices = pd.read_csv(PATH_INDICES, sep=',', encoding='utf-8')

try:
    df_nacional = pd.read_csv(PATH_ING_CHILE, sep=';', encoding='utf-8')
except Exception:
    df_nacional = pd.read_csv(PATH_ING_CHILE, sep=';', encoding='latin-1')

# Estandarización de cabeceras para prevenir fallas por caracteres nulos
if 'Institución' not in df_nacional.columns and df_nacional.columns[0].endswith('Institución'):
    df_nacional.rename(columns={df_nacional.columns[0]: 'Institución'}, inplace=True)


# 3. 📊 PURIFICACIÓN DE LA BASE HISTÓRICA (Para la pestaña GRÁFICOS)
cols_graficos_validas = [
    'Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Vacantes', 
    'Matrícula Primer Año', 'Valor de arancel', 'Matrícula Total',
    'Promedio Puntaje (promedio matemáticas y lenguaje)', 
    'Puntaje de corte (último seleccionado)', 'Promedio Puntaje NEM'
]

for col in cols_graficos_validas:
    if col in df_indices.columns:
        if df_indices[col].dtype == 'object':
            df_indices[col] = df_indices[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).str.strip()
        df_indices[col] = pd.to_numeric(df_indices[col], errors='coerce').astype('float32')

df_indices['Año'] = pd.to_numeric(df_indices['Año'], errors='coerce').fillna(2024).astype('int16')


# 4. 🤖 PURIFICACIÓN DE LA BASE NACIONAL (Para evitar nulos en ML)
def sueldo_parser_auxiliar(texto):
    if pd.isna(texto) or str(texto).lower() == 's/i' or str(texto).strip() == '': 
        return np.nan
    
    t = str(texto).lower()
    
    # Lista limpia: asegúrate de que todos tengan la cantidad correcta de ceros
    if '3 millones 500' in t: return 3500000
    if '3 millones a' in t: return 3200000
    if '2 millones 500' in t: return 2500000
    if '2 millones 400' in t: return 2400000
    if '2 millones 300' in t: return 2300000
    if '2 millones 200' in t: return 2200000
    if '2 millones 100' in t: return 2100000
    if '2 millones' in t: return 2000000
    if '1 millón 900' in t: return 1900000
    if '1 millón 800' in t: return 1800000
    if '1 millón 700' in t: return 1700000
    if '1 millón 600' in t: return 1600000
    if '1 millón 500' in t: return 1500000
    if '1 millón 400' in t: return 1400000
    if '1 millón 300' in t: return 1300000
    if '1 millón 200' in t: return 1200000
    if '1 millón 100' in t: return 1100000
    if '1 millón' in t: return 1000000
    
    return np.nan

cols_ml_validas = ['Empleabilidad al 2º Año', 'Retención de 1er año', 'Duración Real (semestres)', 'Empleabilidad al 1er año']
for col in cols_ml_validas:
    if col in df_nacional.columns:
        if df_nacional[col].dtype == 'object':
            df_nacional[col] = df_nacional[col].astype(str).str.replace('%', '', regex=False).str.replace(',', '.', regex=False).str.strip()
        df_nacional[col] = pd.to_numeric(df_nacional[col], errors='coerce').astype('float32')

if 'Duración Real (semestres)' in df_nacional.columns and df_nacional['Duración Real (semestres)'].max() > 30:
    df_nacional['Duración Real (semestres)'] = df_nacional['Duración Real (semestres)'] / 10.0

if 'Ingreso promedio al 4° año' in df_nacional.columns:
    df_nacional['Ingreso promedio al 4° año'] = df_nacional['Ingreso promedio al 4° año'].apply(sueldo_parser_auxiliar).astype('float32')


print("Generando almacenes de datos unificados...")

# 🔥 SOLUCIÓN: Declaramos ambos nombres en la memoria global de Jupyter
cache_graficos = df_indices.copy()
cache_graficos_indices = df_indices.copy()
cache_kpi_real = df_nacional.copy()

print("➡️ ¡Bases de datos sincronizadas, purificadas y listas para visualización!")
print(f"   - Filas disponibles para entrenamiento ML: {len(cache_kpi_real.dropna(subset=['Ingreso promedio al 4° año']))}")

Cargando matrices de datos en memoria...
Generando almacenes de datos unificados...
➡️ ¡Bases de datos sincronizadas, purificadas y listas para visualización!
   - Filas disponibles para entrenamiento ML: 210


In [ ]:
# Celda 2: Declaracion de contenedores y componentes visuales
import ipywidgets as widgets
from IPython.display import display, clear_output

# Inyeccion de estilos
estilos_css = widgets.HTML("""
<style>
    .tabs-redondeadas .btn {
        border-radius: 16px !important; 
        margin-right: 6px !important;   
        border: 1px solid #bce8f1 !important;
    }
</style>
""")

tabs_navegacion = widgets.ToggleButtons(
    options=['KPIs', 'GRÁFICOS', 'MAPAS', 'ML (RANDOM FOREST PESOS)'],
    value='KPIs', button_style='info', layout=widgets.Layout(width='100%', margin='0px 0px 5px 0px')
)
tabs_navegacion.add_class('tabs-redondeadas') 
linea_separadora = widgets.HTML("<hr style='border: 0; border-top: 3px solid #000000; margin: 5px 0px 15px 0px; width: 100%; opacity: 1;'>")

# MOTOR ESTRICTO: Solo carreras con datos validos (Sin Nulos ni S/I)
# Filtramos primero filas que no sean nulas en las columnas críticas
columnas_criticas = ['Institución', 'Carrera', 'Retención de 1er año', 'Empleabilidad al 1er año', 'Empleabilidad al 2º Año', 'Duración Real (semestres)', 'Ingreso promedio al 4° año']
df_estricto = df_nacional.dropna(subset=columnas_criticas).copy() # type: ignore

# Luego filtramos los textos "s/i" (sin información) que a veces se camuflan como strings
for col in columnas_criticas[2:]:
    df_estricto = df_estricto[~df_estricto[col].astype(str).str.lower().str.contains('s/i')]

opciones_universidades = ['---'] + sorted(df_nacional['Institución'].unique().tolist())

selector_kpi_institucion = widgets.Dropdown(
    options=opciones_universidades, value='---', description='Institucion:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_kpi_carrera = widgets.Dropdown(
    options=['---'], value='---', description='Carrera:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='5px 0px 15px 0px')
)

def actualizar_dropdown_carreras(change):
    institucion_seleccionada = change['new']
    if institucion_seleccionada == '---':
        selector_kpi_carrera.options = ['---']
    else:
        carreras_disponibles = df_nacional[df_nacional['Institución'] == institucion_seleccionada]['Carrera'].unique()
        selector_kpi_carrera.options = ['---'] + sorted(carreras_disponibles.tolist())
    selector_kpi_carrera.value = '---'

selector_kpi_institucion.observe(actualizar_dropdown_carreras, names='value')

# Interfaz KPIs
panel_izquierdo_kpis = widgets.VBox([
    widgets.HTML("<h4>Filtros KPI</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_kpi_institucion, selector_kpi_carrera,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px;'><i>Solo se muestran carreras con data completa (Filtro Estricto).</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb', overflow='hidden'))

area_kpi_cards = widgets.Output(layout=widgets.Layout(width='650px', height='520px', border='1px solid #eee', bg_color='#fafafa', padding='10px', overflow='hidden'))
layout_tab_kpis = widgets.HBox([panel_izquierdo_kpis, area_kpi_cards], layout=widgets.Layout(width='100%', height='100%', padding='0px', overflow='hidden'))

# Interfaz Gráficos
diccionario_graficos = {'---': '', 'Evolución de Puntajes de Selección (Promedio vs Corte)': 'Puntajes', 'Brecha de Género en la Matrícula de Primer Año': 'Género', 'Relación Puntaje NEM vs Puntaje de Selección': 'NEM', 'Comparativa de Vacantes vs Matrícula Efectiva': 'Vacantes'}
selector_graficos = widgets.Dropdown(options=list(diccionario_graficos.keys()), value='---', description='Grafico:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))
panel_izquierdo_graficos = widgets.VBox([
    widgets.HTML("<h4>Reportes Estadísticos</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"), selector_graficos
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb', overflow='hidden'))
area_imagen_grafico = widgets.Output(layout=widgets.Layout(width='650px', height='520px', border='1px solid #eee', bg_color='#fafafa', padding='10px', overflow='hidden'))
layout_tab_graficos = widgets.HBox([panel_izquierdo_graficos, area_imagen_grafico], layout=widgets.Layout(width='100%', height='100%', padding='0px', overflow='hidden'))

# Interfaz Mapas
opciones_mapas = ['---', 'Empleabilidad al 1er Año', 'Retención de Primer Año', 'Mejor Sueldo al 4to Año']
selector_mapas = widgets.Dropdown(options=opciones_mapas, value='---', description='Ver mapa:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))
panel_izquierdo_mapas = widgets.VBox([
    widgets.HTML("<h4>Variables de Distribucion</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"), selector_mapas
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb', overflow='hidden'))
area_imagen_mapa = widgets.Output(layout=widgets.Layout(width='650px', height='520px', border='1px solid #eee', bg_color='#fafafa', padding='0px', overflow='hidden'))
layout_tab_mapas = widgets.HBox([panel_izquierdo_mapas, area_imagen_mapa], layout=widgets.Layout(width='100%', height='100%', padding='0px', overflow='hidden'))

# Interfaz Machine Learning
selector_modelo_ml = widgets.Dropdown(options=['Regresión Lineal', 'K-Means Clustering', 'PCA Análisis'], value='Regresión Lineal', description='Modelo ML:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))
selector_sub_analisis_ml = widgets.Dropdown(options=['---'], value='---', description='Sub-análisis:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))
panel_izquierdo_ml = widgets.VBox([
    widgets.HTML("<h4>Machine Learning</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"), selector_modelo_ml, selector_sub_analisis_ml
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb', overflow='hidden'))

area_imagen_ml = widgets.Output(
    layout=widgets.Layout(width='950px', height='550px', border='1px solid #eee', bg_color='#fafafa', padding='10px', overflow='hidden')
)

layout_tab_ml = widgets.HBox([panel_izquierdo_ml, area_imagen_ml], layout=widgets.Layout(width='100%', height='100%', padding='0px', overflow='hidden'))

contenedor_cuerpo = widgets.Output(layout=widgets.Layout(width='100%', height='540px', overflow='hidden'))
print("Modulos de la interfaz inicializados (Modo Estricto).")

Modulos de la interfaz inicializados (Modo Estricto).


In [3]:
# Celda 3: Procesamiento geoespacial y pre-renderizado HTML local
import os
import json
import pandas as pd
import numpy as np
from IPython.display import HTML, display, clear_output

# Verificacion de la estructura de directorios para cache
CARPETA_MAPAS = 'html_mapas'
if not os.path.exists(CARPETA_MAPAS):
    os.makedirs(CARPETA_MAPAS)

_MAPAS_HTML_CACHE = {}
nombres_archivos = {
    'Empleabilidad al 1er Año': f'{CARPETA_MAPAS}/mapa_emp.html',
    'Retención de Primer Año': f'{CARPETA_MAPAS}/mapa_ret.html',
    'Mejor Sueldo al 4to Año': f'{CARPETA_MAPAS}/mapa_sue.html'
}

faltan_mapas = False

# Lectura directa desde disco para optimizacion de tiempos
for llave, ruta in nombres_archivos.items():
    if os.path.exists(ruta):
        with open(ruta, 'r', encoding='utf-8') as f:
            _MAPAS_HTML_CACHE[llave] = f.read()
    else:
        faltan_mapas = True

# Generacion y simplificacion de poligonos si el cache no existe
if faltan_mapas:
    print("Iniciando calculo geoespacial. Este proceso se ejecutara una sola vez...")
    import geopandas as gpd
    import plotly.graph_objects as go
    import unicodedata
    from shapely.geometry import box as _box

    def _sueldo_a_num(texto):
        if pd.isna(texto) or str(texto).strip().lower() == 's/i': return np.nan
        t = str(texto).lower()
        if 'sobre' in t and '3' in t and '500' in t: return 3750000
        if '3 millones 500' in t: return 3750000
        if '3 millones a' in t: return 3250000
        if '3 millones' in t: return 3250000
        if '2 millones 500' in t: return 2750000
        if '2 millones 400' in t: return 2450000
        if '2 millones 300' in t: return 2350000
        if '2 millones 200' in t: return 2250000
        if '2 millones 100' in t: return 2150000
        if '2 millones' in t: return 2050000
        for val, num in [('900',1950000),('800',1850000),('700',1750000),('600',1650000),('500',1550000),('400',1450000),('300',1350000),('200',1250000),('100',1150000)]:
            if '1 mill' in t and val in t: return num
        if '1 mill' in t: return 1050000
        return np.nan

    _gdf_raw = gpd.read_file('../regiones/Regional.shp').to_crs(epsg=4326)
    
    # Identificacion dinamica de columnas de segmentacion regional
    _col_reg = None
    for _c in ['NOM_REG', 'NOM_REGION', 'REGION', 'nom_reg', 'Region', 'NOMBRE', 'NAME']:
        if _c in _gdf_raw.columns:
            _col_reg = _c
            break
    if _col_reg is None:
        _col_reg = [c for c in _gdf_raw.columns if c != 'geometry'][0]

    _bbox_chile = _box(-76.0, -56.0, -64.0, -17.0)
    _gdf_raw = _gdf_raw.copy()
    _gdf_raw['geometry'] = _gdf_raw.geometry.make_valid()
    _gdf_raw['geometry'] = _gdf_raw['geometry'].intersection(_bbox_chile)
    _gdf_raw['geometry'] = _gdf_raw['geometry'].simplify(tolerance=0.03, preserve_topology=True)
    _gdf_raw = _gdf_raw[~_gdf_raw['geometry'].is_empty].reset_index(drop=True)

    def _norm(s):
        s = unicodedata.normalize('NFKD', str(s).strip().lower()).encode('ascii','ignore').decode()
        return s.replace('-', '').replace('.', '').replace("'", '').replace('  ', ' ').strip()

    _INST_A_REGION = {
        'UNIVERSIDAD DE CHILE': 'Región Metropolitana de Santiago',
        'PONTIFICIA UNIVERSIDAD CATÓLICA DE CHILE': 'Región Metropolitana de Santiago',
        'PONTIFICIA UNIVERSIDAD CATOLICA DE CHILE': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD DE SANTIAGO DE CHILE': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD DIEGO PORTALES': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD ANDRES BELLO': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD ADOLFO IBAÑEZ': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD FINIS TERRAE': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD MAYOR': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD CENTRAL DE CHILE': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD SANTO TOMAS': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD DE LAS AMERICAS': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD ALBERTO HURTADO': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD DEL DESARROLLO': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD AUTONOMA DE CHILE': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD SAN SEBASTIAN': 'Región Metropolitana de Santiago',
        "UNIVERSIDAD BERNARDO O'HIGGINS": 'Región Metropolitana de Santiago',
        'UNIVERSIDAD BERNARDO O´HIGGINS': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD TECNOLOGICA DE CHILE INACAP': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD DE LOS ANDES': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD CATOLICA CARDENAL RAUL SILVA HENRIQUEZ': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD TECNOLOGICA METROPOLITANA': 'Región Metropolitana de Santiago',
        'UNIVERSIDAD DE ARTES, CIENCIAS Y COMUNICACION - UNIACC': 'Región Metropolitana de Santiago',
        'PONTIFICIA UNIVERSIDAD CATÓLICA DE VALPARAÍSO': 'Región de Valparaíso',
        'PONTIFICIA UNIVERSIDAD CATOLICA DE VALPARAISO': 'Región de Valparaíso',
        'UNIVERSIDAD TÉCNICA FEDERICO SANTA MARÍA': 'Región de Valparaíso',
        'UNIVERSIDAD TECNICA FEDERICO SANTA MARIA': 'Región de Valparaíso',
        'UNIVERSIDAD DE VALPARAISO': 'Región de Valparaíso',
        'UNIVERSIDAD DE VIÑA DEL MAR': 'Región de Valparaíso',
        'UNIVERSIDAD DE PLAYA ANCHA DE CIENCIAS DE LA EDUCACION': 'Región de Valparaíso',
        'UNIVERSIDAD DE CONCEPCION': 'Región del Biobío',
        'UNIVERSIDAD DEL BIO-BIO': 'Región del Biobío',
        'UNIVERSIDAD CATOLICA DE LA SANTISIMA CONCEPCION': 'Región del Biobío',
        'UNIVERSIDAD AUSTRAL DE CHILE': 'Región de Los Ríos',
        'UNIVERSIDAD DE LOS LAGOS': 'Región de Los Lagos',
        'UNIVERSIDAD DE LA FRONTERA': 'Región de La Araucanía',
        'UNIVERSIDAD CATOLICA DE TEMUCO': 'Región de La Araucanía',
        'UNIVERSIDAD ADVENTISTA DE CHILE': 'Región de La Araucanía',
        'UNIVERSIDAD DE TALCA': 'Región del Maule',
        'UNIVERSIDAD CATÓLICA DEL MAULE': 'Región del Maule',
        'UNIVERSIDAD CATOLICA DEL MAULE': 'Región del Maule',
        'UNIVERSIDAD DE ANTOFAGASTA': 'Región de Antofagasta',
        'UNIVERSIDAD CATÓLICA DEL NORTE': 'Región de Antofagasta',
        'UNIVERSIDAD CATOLICA DEL NORTE': 'Región de Antofagasta',
        'UNIVERSIDAD DE ATACAMA': 'Región de Atacama',
        'UNIVERSIDAD DE ACONCAGUA': 'Región de Valparaíso',
        'UNIVERSIDAD DE LA SERENA': 'Región de Coquimbo',
        'UNIVERSIDAD DE TARAPACA': 'Región de Arica y Parinacota',
        'UNIVERSIDAD ARTURO PRAT': 'Región de Tarapacá',
    }

    try:
        _df_ing = pd.read_csv('csv_dashboard/todas_las_ingenierias_chile.csv', sep=';', encoding='utf-8')
    except UnicodeDecodeError:
        _df_ing = pd.read_csv('csv_dashboard/todas_las_ingenierias_chile.csv', sep=';', encoding='utf-16')
    _df_ing.columns = [c.strip().lstrip('\ufeff') for c in _df_ing.columns]

    def _pct_float(s):
        return pd.to_numeric(
            s.astype(str).str.replace('%','',regex=False).str.replace(',','.',regex=False).replace('s/i', np.nan),
            errors='coerce'
        )

    _df_ing['Retención de 1er año']     = _pct_float(_df_ing['Retención de 1er año'])
    _df_ing['Empleabilidad al 1er año']  = _pct_float(_df_ing['Empleabilidad al 1er año'])
    _df_ing['Empleabilidad al 2º Año']   = _pct_float(_df_ing['Empleabilidad al 2º Año'])
    _df_ing['Sueldo_Num']                = _df_ing['Ingreso promedio al 4° año'].apply(_sueldo_a_num)
    _df_ing['Region'] = _df_ing['Institución'].str.upper().str.strip().map(
        {k.upper(): v for k, v in _INST_A_REGION.items()}
    )

    rows = []
    for region, grp in _df_ing.groupby('Region'):
        g_emp = grp.dropna(subset=['Empleabilidad al 1er año'])
        g_ret = grp.dropna(subset=['Retención de 1er año'])
        g_sue = grp.dropna(subset=['Sueldo_Num'])
        rows.append({
            'Region':            region,
            'Empleabilidad_1año': grp['Empleabilidad al 1er año'].mean(),
            'Retencion':          grp['Retención de 1er año'].mean(),
            'Sueldo_4año':        grp['Sueldo_Num'].mean(),
            'Mejor_Carrera_Emp':  g_emp.loc[g_emp['Empleabilidad al 1er año'].idxmax(),'Carrera'] if not g_emp.empty else 'N/D',
            'Mejor_Emp_Val':      g_emp['Empleabilidad al 1er año'].max() if not g_emp.empty else np.nan,
            'Mejor_Carrera_Ret':  g_ret.loc[g_ret['Retención de 1er año'].idxmax(),'Carrera'] if not g_ret.empty else 'N/D',
            'Mejor_Ret_Val':      g_ret['Retención de 1er año'].max() if not g_ret.empty else np.nan,
            'Mejor_Carrera_Sue':  g_sue.loc[g_sue['Sueldo_Num'].idxmax(),'Carrera'] if not g_sue.empty else 'N/D',
            'Mejor_Sue_Val':      g_sue['Sueldo_Num'].max() if not g_sue.empty else np.nan,
            'N_Carreras':         grp['Carrera'].nunique(),
            'N_Inst':             grp['Institución'].nunique(),
        })
    _df_kpi = pd.DataFrame(rows)

    _gdf_raw['_key'] = _gdf_raw[_col_reg].apply(_norm)
    _df_kpi['_key']  = _df_kpi['Region'].apply(_norm)
    _gdf_merged = _gdf_raw.merge(_df_kpi, on='_key', how='left').reset_index(drop=True)

    _PRECALCULATED_GEOJSON = json.loads(_gdf_merged.to_json())

    # Funcion maestra de renderizacion georeferenciada
    def _crear_mapa_html(col_z, titulo, colorscale, es_sueldo=False, col_carrera=None, col_mejor_val=None, etiqueta=''):
        geojson  = _PRECALCULATED_GEOJSON
        ids_str  = [str(i) for i in _gdf_merged.index]
        valores  = _gdf_merged[col_z].tolist()

        hover_texts = []
        for _, row in _gdf_merged.iterrows():
            nombre = str(row.get(_col_reg, '')).strip()
            val    = row[col_z]
            if pd.isna(val):
                hover_texts.append(f"<b>{nombre}</b><br><i>Sin registros</i>")
                continue
            val_str = f"${int(val):,}".replace(',','.') if es_sueldo else f"{val:.1f}%"
            txt = f"<b>{nombre}</b><br>{titulo}: <b>{val_str}</b>"
            if col_carrera and pd.notna(row.get(col_carrera)):
                mv   = row.get(col_mejor_val, np.nan)
                mv_s = (f"${int(mv):,}".replace(',','.') if es_sueldo else f"{mv:.1f}%") if pd.notna(mv) else ''
                txt += f"<br><br>Lider en {etiqueta}:<br><i>{row[col_carrera]}</i>"
                if mv_s: txt += f"  ({mv_s})"
            if pd.notna(row.get('N_Carreras')):
                txt += f"<br><br>Programas: {int(row['N_Carreras'])} | Entidades: {int(row['N_Inst'])}"
            hover_texts.append(txt)

        vals_ok = [v for v in valores if v is not None and not (isinstance(v,float) and np.isnan(v))]

        if es_sueldo:
            r_min = float(min(vals_ok)) * 0.9 if vals_ok else 0
            r_max = float(max(vals_ok)) * 1.05 if vals_ok else 1000000
        else:
            r_min = float(min(vals_ok)) - 4 if vals_ok else 0
            r_max = float(max(vals_ok)) + 4 if vals_ok else 100

        fig = go.Figure(go.Choropleth(
            geojson      = geojson,
            locations    = ids_str,
            z            = valores,
            featureidkey = 'id',
            colorscale   = colorscale,
            colorbar     = dict(
                title      = dict(text=titulo, font=dict(size=11)),
                thickness  = 20, 
                len        = 0.7, 
                x          = 1.15,
                xanchor    = 'left',
                tickmode   = 'auto',
                nticks     = 10,  
                tickformat = '$,.0f' if es_sueldo else '.1f',
            ),
            hovertext        = hover_texts,
            hoverinfo        = 'text',
            marker_line_color= '#888888', 
            marker_line_width= 0.8,
            zmin = r_min,
            zmax = r_max,
        ))

        fig.update_geos(
            visible          = False,
            showland         = True, 
            landcolor        = '#e0e0e0', 
            fitbounds        = "locations", 
        )
        
        fig.update_layout(
            title         = dict(text=titulo, x=0.5, font=dict(size=14, color='#2c3e50')),
            height        = 520, 
            width         = 620,
            margin        = dict(l=50, r=120, t=50, b=50),
            paper_bgcolor = 'rgba(0,0,0,0)',
            plot_bgcolor  = 'rgba(0,0,0,0)',
        )
        return fig.to_html(include_plotlyjs='cdn', full_html=False)

    # Escritura de resultados estructurados en disco
    _MAPAS_HTML_CACHE['Empleabilidad al 1er Año'] = _crear_mapa_html('Empleabilidad_1año', 'Empleabilidad 1er Año (%)', 'RdYlGn', col_carrera='Mejor_Carrera_Emp', col_mejor_val='Mejor_Emp_Val', etiqueta='Empleabilidad')
    with open(nombres_archivos['Empleabilidad al 1er Año'], 'w', encoding='utf-8') as f: f.write(_MAPAS_HTML_CACHE['Empleabilidad al 1er Año'])

    _MAPAS_HTML_CACHE['Retención de Primer Año'] = _crear_mapa_html('Retencion', 'Retencion 1er Año (%)', 'Blues', col_carrera='Mejor_Carrera_Ret', col_mejor_val='Mejor_Ret_Val', etiqueta='Retencion')
    with open(nombres_archivos['Retención de Primer Año'], 'w', encoding='utf-8') as f: f.write(_MAPAS_HTML_CACHE['Retención de Primer Año'])

    _MAPAS_HTML_CACHE['Mejor Sueldo al 4to Año'] = _crear_mapa_html('Sueldo_4año', 'Sueldo Promedio 4to Año', 'YlOrRd', es_sueldo=True, col_carrera='Mejor_Carrera_Sue', col_mejor_val='Mejor_Sue_Val', etiqueta='Remuneracion')
    with open(nombres_archivos['Mejor Sueldo al 4to Año'], 'w', encoding='utf-8') as f: f.write(_MAPAS_HTML_CACHE['Mejor Sueldo al 4to Año'])

    print("Mapas procesados y compilados en directorio local.")
else:
    print("Directorio html_mapas detectado. Omitiendo calculo geoespacial.")

Directorio html_mapas detectado. Omitiendo calculo geoespacial.


In [4]:
# Celda 4: Motores Reactivos, Filtros y Enrutador
import sys
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
import plotly.express as px

sys.path.insert(0, '.')  
from rlm import entrenar_regresion_ingresos
from kmeans import clustering_carreras
from pca import analisis_pca_completo

ANCHO_FIJO = '1300px'
ALTO_FIJO = '650px'

# --- 1. FUNCIÓN AUXILIAR DE SUELDOS MEJORADA (REGEX) ---
def sueldo_a_numero(texto):
    if pd.isna(texto): return None
    t = str(texto).lower().strip()
    if t in ['s/i', 'sin información', 'sin informacion', '-', '']: return None
    
    # 1. Chequeo de formato texto Mi Futuro
    if '3 millones 500' in t: return 3750000
    if '3 millones a' in t: return 3250000
    if '2 millones 500' in t: return 2750000
    if '2 millones 400' in t: return 2450000
    if '2 millones 300' in t: return 2350000
    if '2 millones 200' in t: return 2250000
    if '2 millones 100' in t: return 2150000
    if '2 millones' in t: return 2050000
    if '1 millón 900' in t: return 1950000
    if '1 millón 800' in t: return 1850000
    if '1 millón 700' in t: return 1750000
    if '1 millón 600' in t: return 1650000
    if '1 millón 500' in t: return 1550000
    if '1 millón 400' in t: return 1450000
    if '1 millón 300' in t: return 1350000
    if '1 millón 200' in t: return 1250000
    if '1 millón 100' in t: return 1150000
    if '1 millón' in t: return 1050000
    
    # 2. Extractor Universal Regex (Maneja "$1.500.000", "1500000", o rangos "1.200.000 a 1.300.000")
    t_limpio = t.replace('.', '')
    numeros = re.findall(r'\d+', t_limpio)
    if numeros:
        valores = [float(n) for n in numeros if float(n) > 50000] # Evitar números residuales pequeños
        if valores:
            return sum(valores) / len(valores) # Promedia si es un rango
            
    return None

# --- 2. ENRUTADOR PRINCIPAL DE PESTAÑAS ---
def alternar_pestanas(change):
    with contenedor_cuerpo:
        clear_output(wait=True)
        pestana_activa = change['new'] if change else tabs_navegacion.value
        
        if pestana_activa == 'KPIs':
            display(layout_tab_kpis)
            try: selector_kpi_institucion.unobserve(actualizar_kpi_cards, names='value')
            except ValueError: pass
            try: selector_kpi_carrera.unobserve(actualizar_kpi_cards, names='value')
            except ValueError: pass
            selector_kpi_institucion.value = '---'
            selector_kpi_carrera.value = '---'
            selector_kpi_institucion.observe(actualizar_kpi_cards, names='value')
            selector_kpi_carrera.observe(actualizar_kpi_cards, names='value')
            actualizar_kpi_cards(None)
            
        elif pestana_activa == 'GRÁFICOS':
            display(layout_tab_graficos)
            try: selector_graficos.unobserve(actualizar_imagen_grafico, names='value')
            except ValueError: pass
            selector_graficos.value = '---'
            selector_graficos.observe(actualizar_imagen_grafico, names='value')
            actualizar_imagen_grafico(None)
            
        elif pestana_activa == 'MAPAS':
            display(layout_tab_mapas)
            try: selector_mapas.unobserve(_renderizar_mapa, names='value')
            except ValueError: pass
            selector_mapas.value = '---'
            selector_mapas.observe(_renderizar_mapa, names='value')
            _renderizar_mapa(None)
            
        elif pestana_activa == 'ML (RANDOM FOREST PESOS)':
            display(layout_tab_ml)
            try: selector_modelo_ml.unobserve(actualizar_opciones_sub_analisis, names='value')
            except ValueError: pass
            try: selector_sub_analisis_ml.unobserve(actualizar_sub_analisis_ml, names='value')
            except ValueError: pass
            selector_modelo_ml.value = 'Regresión Lineal'
            selector_sub_analisis_ml.options = ['---']
            selector_sub_analisis_ml.value = '---'
            selector_modelo_ml.observe(actualizar_opciones_sub_analisis, names='value')
            selector_sub_analisis_ml.observe(actualizar_sub_analisis_ml, names='value')
            actualizar_sub_analisis_ml(None) 

tabs_navegacion.observe(alternar_pestanas, names='value')


# --- 3. MOTOR DE MACHINE LEARNING ---
def actualizar_opciones_sub_analisis(change):
    modelo = selector_modelo_ml.value
    if modelo == 'Regresión Lineal': opciones = ['Predicción vs Actual', 'Análisis Residuales', 'Importancia Features']
    elif modelo == 'K-Means Clustering': opciones = ['Elbow Method', 'Clusters 2D (PCA)', 'Clusters 3D (PCA)']
    elif modelo == 'PCA Análisis': opciones = ['Scree Plot', 'Biplot 2D', 'Loading Heatmap', 'PCA 3D']
    else: opciones = ['---']
    
    selector_sub_analisis_ml.unobserve(actualizar_sub_analisis_ml, names='value')
    selector_sub_analisis_ml.options = opciones
    selector_sub_analisis_ml.value = opciones[0] if opciones else '---'
    selector_sub_analisis_ml.observe(actualizar_sub_analisis_ml, names='value')
    actualizar_sub_analisis_ml(None)

selector_modelo_ml.observe(actualizar_opciones_sub_analisis, names='value')

def actualizar_sub_analisis_ml(change):
    with area_imagen_ml:
        clear_output(wait=True)
        modelo = selector_modelo_ml.value
        sub_analisis = selector_sub_analisis_ml.value
        if sub_analisis == '---' or not sub_analisis: return
            
        if modelo == 'Regresión Lineal':
            resultado_regresion = entrenar_regresion_ingresos(cache_kpi_real)
            if resultado_regresion['error']: display(widgets.HTML(f"<div style='padding:20px; color:#d9534f;'><b>❌ Error:</b> {resultado_regresion['error']}</div>"))
            else:
                metricas = resultado_regresion['metricas']
                display(widgets.HTML(f"""<div style='background-color:#f0f8ff; border-left:4px solid #5cb85c; padding:10px; margin-bottom:10px; font-size:0.9em; font-family:monospace;'><b>Métricas:</b> R²={metricas.get('R² Test', 0):.4f} | RMSE=${metricas.get('RMSE Test', 0):,.0f} | MAE=${metricas.get('MAE Test', 0):,.0f}</div>"""))
                if sub_analisis == 'Predicción vs Actual': display(resultado_regresion['figura_prediccion'])
                elif sub_analisis == 'Análisis Residuales': display(resultado_regresion['figura_residuales'])
                elif sub_analisis == 'Importancia Features': display(resultado_regresion['figura_importancia'])
        
        elif modelo == 'K-Means Clustering':
            resultado_kmeans = clustering_carreras(cache_kpi_real)
            if resultado_kmeans['error']: display(widgets.HTML(f"<div style='padding:20px; color:#d9534f;'><b>❌ Error:</b> {resultado_kmeans['error']}</div>"))
            else:
                metricas = resultado_kmeans['metricas']
                display(widgets.HTML(f"""<div style='background-color:#fffacd; border-left:4px solid #f0ad4e; padding:10px; margin-bottom:10px; font-size:0.9em; font-family:monospace;'><b>Clustering:</b> K={metricas.get('K_optimo', 0)} | Varianza 2D={metricas.get('varianza_explicada_2d', 0):.1f}%</div>"""))
                if sub_analisis == 'Elbow Method': display(resultado_kmeans['figura_elbow'])
                elif sub_analisis == 'Clusters 2D (PCA)': display(resultado_kmeans['figura_clusters_2d'])
                elif sub_analisis == 'Clusters 3D (PCA)': display(resultado_kmeans['figura_clusters_3d'])
        
        elif modelo == 'PCA Análisis':
            resultado_pca = analisis_pca_completo(cache_kpi_real)
            if resultado_pca['error']: display(widgets.HTML(f"<div style='padding:20px; color:#d9534f;'><b>❌ Error:</b> {resultado_pca['error']}</div>"))
            else:
                metricas = resultado_pca['metricas']
                display(widgets.HTML(f"""<div style='background-color:#f0f8f8; border-left:4px solid #5bc0de; padding:10px; margin-bottom:10px; font-size:0.9em; font-family:monospace;'><b>PCA:</b> Componentes={metricas.get('n_componentes_80_varianza', 0)} | Varianza 3D={metricas.get('varianza_explicada_3d', 0):.1f}%</div>"""))
                if sub_analisis == 'Scree Plot': display(resultado_pca['figura_scree'])
                elif sub_analisis == 'Biplot 2D': display(resultado_pca['figura_biplot'])
                elif sub_analisis == 'Loading Heatmap': display(resultado_pca['figura_loadings_heatmap'])
                elif sub_analisis == 'PCA 3D': display(resultado_pca['figura_pca_3d'])

selector_sub_analisis_ml.observe(actualizar_sub_analisis_ml, names='value')


# --- 4. MOTOR DINÁMICO DE KPIs ---
def actualizar_kpi_cards(change):
    with area_kpi_cards:
        clear_output(wait=True)
        institucion = selector_kpi_institucion.value
        carrera = selector_kpi_carrera.value
        
        if institucion == '---' or carrera == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Seleccione parametros validos para iniciar el reporte numerico ]</h4></div>"))
            return
            
        df_res = cache_kpi_real[(cache_kpi_real['Institución'] == institucion) & (cache_kpi_real['Carrera'] == carrera)].copy()
        
        if not df_res.empty:
            for col in ['Retención de 1er año', 'Empleabilidad al 1er año', 'Empleabilidad al 2º Año']:
                if col in df_res.columns:
                    df_res[col] = df_res[col].astype(str).str.replace('%', '', regex=False).str.replace(',', '.', regex=False)
                    df_res[col] = pd.to_numeric(df_res[col], errors='coerce')
            
            if 'Duración Real (semestres)' in df_res.columns:
                df_res['Duración Real (semestres)'] = pd.to_numeric(df_res['Duración Real (semestres)'], errors='coerce')
            
            if 'Ingreso promedio al 4° año' in df_res.columns:
                df_res['Sueldo_Calculado'] = df_res['Ingreso promedio al 4° año']
            else:
                df_res['Sueldo_Calculado'] = np.nan

            mean_ret = df_res['Retención de 1er año'].dropna().mean() if 'Retención de 1er año' in df_res.columns else np.nan
            v_ret = f"{mean_ret:.1f}%" if pd.notna(mean_ret) else "--%"
            
            emp_cols = [c for c in ['Empleabilidad al 1er año', 'Empleabilidad al 2º Año'] if c in df_res.columns]
            if emp_cols:
                all_emp = pd.concat([df_res[c] for c in emp_cols]).dropna()
                mean_emp = all_emp.mean() if not all_emp.empty else np.nan
            else:
                mean_emp = np.nan
            v_emp = f"{mean_emp:.1f}%" if pd.notna(mean_emp) else "--%"
            
            mean_dur_semestres = df_res['Duración Real (semestres)'].dropna().mean() if 'Duración Real (semestres)' in df_res.columns else np.nan
            if pd.notna(mean_dur_semestres):
                dur_real_anos = mean_dur_semestres / 2.0
                dur_teorica = 5.0 if 'comercial' in str(carrera).lower() else 5.5
                sobreduracion = max(0.0, dur_real_anos - dur_teorica)
                v_dur = f"+{sobreduracion:.1f} años"
            else:
                v_dur = "-- años"
            
            mean_ingreso = df_res['Sueldo_Calculado'].dropna().mean()
            v_ingreso = f"${int(mean_ingreso):,}".replace(',', '.') if pd.notna(mean_ingreso) else "No disponible"
        else:
            v_ret, v_emp, v_dur, v_ingreso = "--%", "--%", "-- años", "No disponible"
            
        html_content = f"""
        <div style='font-family: sans-serif; padding: 5px; height:100%;'>
            <h4 style='color: #2c3e50; margin-top: 0; margin-bottom: 5px;'>Promedios Globales Unificados: {institucion}</h4>
            <h5 style='color: #555; margin-top: 0; margin-bottom: 15px; font-weight: normal;'>Programa: <b>{carrera}</b></h5>
            <div style='display: flex; justify-content: space-between; gap: 10px; margin-bottom: 15px;'>
                <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5bc0de; padding: 10px; border-radius: 4px; text-align: center;'>
                    <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Retención</div>
                    <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_ret}</div>
                    <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Promedio histórico</div>
                </div>
                <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5cb85c; padding: 10px; border-radius: 4px; text-align: center;'>
                    <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Empleabilidad</div>
                    <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_emp}</div>
                    <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Media de egreso</div>
                </div>
                <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #f0ad4e; padding: 10px; border-radius: 4px; text-align: center;'>
                    <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Sobreduración</div>
                    <div style='font-size: 1.4em; font-weight: bold; color: #d9534f; margin-top:4px;'>{v_dur}</div>
                    <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Diferencial académico</div>
                </div>
            </div>
            <div style='background-color: #eef9f0; border: 1px solid #c3e6cb; border-left: 6px solid #28a745; padding: 12px; border-radius: 4px; margin-bottom: 15px; text-align: center;'>
                <div style='font-size: 0.85em; color: #155724; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;'>Sueldo Promedio Unificado de la Disciplina</div>
                <div style='font-size: 1.8em; font-weight: bold; color: #1e7e34; margin-top: 5px;'>{v_ingreso}</div>
            </div>
        </div>
        """
        display(widgets.HTML(html_content))

selector_kpi_institucion.observe(actualizar_kpi_cards, names='value')
selector_kpi_carrera.observe(actualizar_kpi_cards, names='value')


# --- 5. MOTOR DE REPORTE DE GRÁFICOS HISTÓRICOS ---
def actualizar_imagen_grafico(change):
    with area_imagen_grafico:
        clear_output(wait=True)
        opcion_seleccionada = selector_graficos.value
        
        if opcion_seleccionada == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'><h4>[ Visualizador en espera de ordenes de renderizado ]</h4></div>"))
        else:
            df_working = cache_graficos.copy()
            if opcion_seleccionada == 'Evolución de Puntajes de Selección (Promedio vs Corte)':
                df_g = df_working.groupby('Año')[['Promedio Puntaje (promedio matemáticas y lenguaje)', 'Puntaje de corte (último seleccionado)']].mean().reset_index()
                fig = px.line(df_g, x='Año', y=['Promedio Puntaje (promedio matemáticas y lenguaje)', 'Puntaje de corte (último seleccionado)'], markers=True, title='Comportamiento de corte admisión')
            elif opcion_seleccionada == 'Brecha de Género en la Matrícula de Primer Año':
                df_g = df_working.groupby('Año')[['Matrícula primer año hombres', 'Matrícula primer año mujeres']].sum().reset_index()
                df_g['Porcentaje Mujeres (%)'] = (df_g['Matrícula primer año mujeres'] / (df_g['Matrícula primer año hombres'] + df_g['Matrícula primer año mujeres'])) * 100
                fig = px.line(df_g, x='Año', y='Porcentaje Mujeres (%)', markers=True, title='Fluctuación demográfica de género')
                fig.update_yaxes(range=[0, 60])
            elif opcion_seleccionada == 'Relación Puntaje NEM vs Puntaje de Selección':
                fig = px.scatter(df_working, x='Promedio Puntaje NEM', y='Promedio Puntaje (promedio matemáticas y lenguaje)', color='Carrera Genérica', hover_data=['Año'], title='Distribución NEM vs Criterio de ingreso')
            elif opcion_seleccionada == 'Comparativa de Vacantes vs Matrícula Efectiva':
                df_g = df_working.groupby('Año')[['Vacantes', 'Matrícula Primer Año']].sum().reset_index()
                fig = px.bar(df_g, x='Año', y=['Vacantes', 'Matrícula Primer Año'], barmode='group', title='Capacidad operacional vs Demanda')
            else:
                display(widgets.HTML("<div style='padding:20px;'><h4>Reporte No Soportado</h4></div>"))
                return

            fig.update_layout(
                height=480, width=600, margin=dict(l=10, r=20, t=50, b=65), 
                paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', showlegend=False,
                updatemenus=[dict(type="buttons", direction="right", x=0.95, y=-0.15, xanchor='right', yanchor='top', showactive=False,
                        buttons=list([dict(label="Mostrar Info", method="relayout", args=[{"showlegend": True}]), dict(label="Ocultar Info", method="relayout", args=[{"showlegend": False}])]))]
            )
            fig.show()

selector_graficos.observe(actualizar_imagen_grafico, names='value')


# --- 6. SUBSISTEMA DE VISOR DE MAPAS REGIONALES ---
def _renderizar_mapa(change):
    with area_imagen_mapa:
        clear_output(wait=False)
        opcion = selector_mapas.value
        if opcion == '---' or opcion is None:
            display(widgets.HTML("<div style='text-align:center; padding-top:90px;'><h4 style='color:#bbb; margin:12px 0 8px 0;'>[ Reporte Regional ] Sistema listo</h4></div>"))
            return
        if '_MAPAS_HTML_CACHE' in globals() and opcion in _MAPAS_HTML_CACHE: display(HTML(_MAPAS_HTML_CACHE[opcion]))
        else: display(widgets.HTML("<b>Error en subsistema de renderizado: Archivo o caché inexistente.</b>"))

try: selector_mapas.unobserve(_renderizar_mapa, names='value')
except Exception: pass
selector_mapas.observe(_renderizar_mapa, names='value')


# --- 7. MAQUETACIÓN E INFRAESTRUCTURA DE LOGEO FINAL ---
txt_usuario = widgets.Text(description='Usuario:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
txt_password = widgets.Password(description='Clave:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
btn_login = widgets.Button(description='Autenticar', button_style='primary', icon='lock', layout=widgets.Layout(margin='20px 0px 5px 0px', width='280px'))
html_feedback = widgets.HTML(value="")

formulario_interno = widgets.VBox([
    widgets.HTML("<h3 style='text-align: center; font-family: sans-serif; color: #333; margin-top:0;'>CONTROL DE ACCESO</h3><hr style='width: 100%; border: 0; border-top: 1px solid #ccc;'>"),
    txt_usuario, txt_password, btn_login, html_feedback
], layout=widgets.Layout(width='360px', padding='25px', border='1px solid #ccc', bg_color='#ffffff', align_items='center', border_radius='4px'))

cuadro_login = widgets.VBox([formulario_interno], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', justify_content='center', align_items='center', overflow='hidden'))
dashboard_final = widgets.VBox([estilos_css, tabs_navegacion, linea_separadora, contenedor_cuerpo], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', padding='20px', overflow='hidden'))

def validar_credenciales(b):
    if txt_usuario.value == 'admin' and txt_password.value == 'admin':
        with lienzo_maestro:
            clear_output()
            display(dashboard_final)
            alternar_pestanas(None)
    else:
        txt_password.value = ""
        html_feedback.value = "<div style='color: #d9534f; font-weight: bold; text-align: center; margin-top: 12px; font-family: sans-serif;'>Excepcion: Credenciales invalidas</div>"

btn_login.on_click(validar_credenciales)
def _on_enter_login(widget, content, buffers):
    if content.get('event', '') == 'keydown' and content.get('key', '') == 'Enter': validar_credenciales(None)
txt_usuario.on_msg(_on_enter_login)
txt_password.on_msg(_on_enter_login)

lienzo_maestro = widgets.Output(layout=widgets.Layout(overflow='hidden'))

# --- 8. EJECUCIÓN DEL DESPLIEGUE INICIAL ---
display(lienzo_maestro)
with lienzo_maestro:
    clear_output()
    display(cuadro_login)

Output(layout=Layout(overflow='hidden'))

In [5]:
# %% Sueldos, Retención, Duraciones y las dos Empleabilidades
import pandas as pd
import numpy as np

features_ml = [
    'Empleabilidad al 2º Año',
    'Retención de 1er año',
    'Duración Real (semestres)',
    'Ingreso promedio al 4° año',
    'Empleabilidad al 1er año'
]

print("=== REVISIÓN DE COLUMNAS EN DF_NACIONAL ===")
print(f"Total de filas iniciales: {len(df_nacional)}\n")

for col in features_ml:
    if col in df_nacional.columns:
        # Evaluamos cuántos nulos reales o valores con formato roto tiene
        valores_vivos = pd.to_numeric(df_nacional[col], errors='coerce')
        nulos_post_conversion = valores_vivos.isna().sum()
        
        print(f"🔹 Columna: '{col}'")
        print(f"   - Tipo de dato original: {df_nacional[col].dtype}")
        print(f"   - Filas que se transformarán en NaN: {nulos_post_conversion} de {len(df_nacional)}")
        print(f"   - Muestra de valores crudos: {list(df_nacional[col].head(3))}")
    else:
        print(f"❌ ¡ALERTA! La columna '{col}' NO existe en el CSV. Revisa si hay pifias en tildes, espacios o el símbolo º/°.")
    print("-" * 50)

# Ver el resultado final tras el cruce dropna
df_test = df_nacional[features_ml].copy()
for col in df_test.columns:
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')
df_test_clean = df_test.dropna()
print(f"\n➡️ RESULTADO FINAL DEL DROPNA: Quedan {len(df_test_clean)} filas con datos completos para el modelo.")

=== REVISIÓN DE COLUMNAS EN DF_NACIONAL ===
Total de filas iniciales: 306

🔹 Columna: 'Empleabilidad al 2º Año'
   - Tipo de dato original: float32
   - Filas que se transformarán en NaN: 32 de 306
   - Muestra de valores crudos: [92.5, 84.19999694824219, 90.0]
--------------------------------------------------
🔹 Columna: 'Retención de 1er año'
   - Tipo de dato original: float32
   - Filas que se transformarán en NaN: 74 de 306
   - Muestra de valores crudos: [88.5999984741211, 75.0999984741211, 81.0]
--------------------------------------------------
🔹 Columna: 'Duración Real (semestres)'
   - Tipo de dato original: float32
   - Filas que se transformarán en NaN: 94 de 306
   - Muestra de valores crudos: [13.399999618530273, 11.600000381469727, 10.0]
--------------------------------------------------
🔹 Columna: 'Ingreso promedio al 4° año'
   - Tipo de dato original: float32
   - Filas que se transformarán en NaN: 96 de 306
   - Muestra de valores crudos: [nan, 2000000.0, nan]
------